# Structured Data Extraction from Invoices

**Problem:** Pulling invoice fields out of plain-text invoices into a clean JSON schema is the kind of glue work that's death-by-a-thousand-cuts manually. I wanted to see how far an LLM plus a Pydantic schema gets you on a small batch.

**Data:** 30 paired files in `../../data/invoices/`. For each `INV-NNN` there's a `.txt` (the unstructured invoice the model sees) and a `.json` (ground truth fields). Synthetic, no real PII.

**Approach:** LLM extraction, then Pydantic parse. If validation fails, retry once with the error pasted back into the prompt. I compute field-level precision/recall against the ground-truth JSON inside the notebook so there's nothing cached.

**Originally I wanted to LoRA-tune Mistral-7B on this**, but the zero-shot model is already strong on these templated invoices, so fine-tuning would mostly be overfitting to the template. Notes at the bottom.

In [1]:
# !pip install transformers peft accelerate datasets pydantic fastapi -q

In [2]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, field_validator
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

DATA_DIR = Path("../../data/invoices")
assert DATA_DIR.exists(), "Drop synthetic invoices into data/invoices/ (one .txt + .json per ID)."

## Loading the invoices

In [3]:
# For each INV-XXXX I want both the .txt (input) and the .json (ground truth).
pairs = []
for txt_path in sorted(DATA_DIR.glob("*.txt")):
    json_path = txt_path.with_suffix(".json")
    if not json_path.exists():
        continue
    with open(txt_path) as fp:
        text = fp.read()
    with open(json_path) as fp:
        gt = json.load(fp)
    pairs.append({"id": txt_path.stem, "text": text, "ground_truth": gt})

print(f"Paired (txt + json) invoices: {len(pairs)}")
if pairs:
    print("Sample ground-truth keys:", list(pairs[0]["ground_truth"].keys()))
    print("\nSample invoice text:")
    print(pairs[0]["text"][:300])

Paired (txt + json) invoices: 30
Sample ground-truth keys: ['invoice_number', 'vendor_name', 'vendor_address', 'customer_name', 'customer_address', 'invoice_date', 'due_date', 'line_items', 'subtotal', 'tax', 'total', 'currency']

Sample invoice text:
INVOICE INV-2024000
Date: 2026-03-04
From: Oconnell PLC
To: Massey-Perry

  Maximize Dynamic E-Markets: 5 x $114.79 = $573.95
  Reinvent Wireless Functionalities: 2 x $82.75 = $165.5

Subtotal: $739.45
Tax: $59.16
Total: $798.61



## Pydantic schema for the invoice fields

In [4]:
class LineItem(BaseModel):
    description: str
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    total: Optional[float] = None

class InvoiceExtraction(BaseModel):
    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    vendor_name: Optional[str] = None
    vendor_address: Optional[str] = None
    customer_name: Optional[str] = None
    customer_address: Optional[str] = None
    due_date: Optional[str] = None
    subtotal: Optional[float] = None
    tax: Optional[float] = None
    total: Optional[float] = None        # matches the ground-truth key name
    currency: Optional[str] = None
    line_items: List[LineItem] = []

    @field_validator("subtotal", "tax", "total", mode="before")
    @classmethod
    def parse_currency(cls, v):
        # the model sometimes hands back "$739.45" instead of a number
        if isinstance(v, str):
            return float(v.replace("$", "").replace(",", "").strip())
        return v

print("Schema fields:", list(InvoiceExtraction.model_fields.keys()))

Schema fields: ['invoice_number', 'invoice_date', 'vendor_name', 'vendor_address', 'customer_name', 'customer_address', 'due_date', 'subtotal', 'tax', 'total', 'currency', 'line_items']


## Modeling: extraction with Pydantic validation and a retry loop

I'm using `gemini-flash-lite-latest` through the OpenAI-compatible endpoint. The retry loop pastes the validation error back into the prompt and asks the model to fix it.

In [5]:
def _strip_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s

# I'm passing the field names and types as a shape hint instead of a full JSON-Schema dict.
# The model handles this much more reliably (I tried JSON-Schema first, it kept hallucinating extra fields).
SCHEMA_HINT = """{
  "invoice_number": str, "invoice_date": "YYYY-MM-DD", "vendor_name": str,
  "vendor_address": str, "customer_name": str, "customer_address": str,
  "due_date": "YYYY-MM-DD", "subtotal": number, "tax": number, "total": number,
  "currency": str,
  "line_items": [{"description": str, "quantity": number, "unit_price": number, "total": number}]
}"""

EXTRACT_PROMPT = """Extract structured invoice data from the text below. Return JSON matching this shape:

{schema}

If a field isn't present, omit it or use null. Numbers should be numbers, not strings.

Invoice text:
{text}
"""

def extract_invoice(text, max_retries=3):
    last_err = None
    for attempt in range(max_retries):
        # On a retry, include the previous error so the model can correct itself.
        prompt = EXTRACT_PROMPT.format(schema=SCHEMA_HINT, text=text[:3000])
        if last_err:
            prompt += f"\n\nPrevious attempt failed validation with: {last_err}\nFix and retry."
        resp = client.chat.completions.create(
            model="gemini-flash-lite-latest",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        raw = _strip_fences(resp.choices[0].message.content)
        try:
            parsed = InvoiceExtraction.model_validate_json(raw)
            return parsed, attempt + 1
        except Exception as e:
            last_err = str(e)[:300]
    return None, max_retries

results = []
N = min(20, len(pairs))
for p in pairs[:N]:
    extracted, attempts = extract_invoice(p["text"])
    if extracted is not None:
        results.append({
            "id": p["id"],
            "extracted": extracted.model_dump(),
            "attempts": attempts,
            "ground_truth": p["ground_truth"],
        })

if results:
    print(f"Extracted {len(results)}/{N} invoices")
    print(f"Avg attempts to pass validation: {np.mean([r['attempts'] for r in results]):.2f}")
    print(f"Needed at least one retry: {sum(r['attempts'] > 1 for r in results)}/{len(results)}")
else:
    print("Nothing passed validation. Check the schema / prompt.")

Extracted 20/20 invoices
Avg attempts to pass validation: 1.00
Needed at least one retry: 0/20


## Evaluation: field-level precision and recall

In [6]:
# Compare on the scalar keys that ground truth actually uses.
SCALAR_FIELDS = ["invoice_number", "invoice_date", "vendor_name", "customer_name",
                 "due_date", "subtotal", "tax", "total"]

def normalize(v):
    if v is None:
        return None
    if isinstance(v, float):
        return round(v, 2)
    return str(v).strip().lower()

field_stats = {f: {"tp": 0, "fp": 0, "fn": 0, "tn": 0} for f in SCALAR_FIELDS}

for r in results:
    ext = r["extracted"]
    gt  = r["ground_truth"]
    for field in SCALAR_FIELDS:
        pred = normalize(ext.get(field))
        true = normalize(gt.get(field))
        if pred is not None and true is not None:
            if pred == true:
                field_stats[field]["tp"] += 1
            else:
                # Wrong extraction = both a wrong prediction (fp) and a missed correct value (fn).
                # Strict interpretation, but the alternative double-counts in a confusing way.
                field_stats[field]["fp"] += 1
                field_stats[field]["fn"] += 1
        elif pred is None and true is not None:
            field_stats[field]["fn"] += 1
        elif pred is not None and true is None:
            field_stats[field]["fp"] += 1
        else:
            field_stats[field]["tn"] += 1

eval_rows = []
for field, s in field_stats.items():
    p = s["tp"] / max(s["tp"] + s["fp"], 1)
    rec = s["tp"] / max(s["tp"] + s["fn"], 1)
    eval_rows.append({
        "field": field,
        "support": s["tp"] + s["fn"],
        "precision": round(p, 3),
        "recall":    round(rec, 3),
    })

eval_df = pd.DataFrame(eval_rows)
print(f"\nField-level metrics over {len(results)} invoices:\n")
print(eval_df.to_string(index=False))

# Quick look at one mismatch so the failure mode is concrete.
for r in results:
    for f in SCALAR_FIELDS:
        if normalize(r["extracted"].get(f)) != normalize(r["ground_truth"].get(f)) \
                and r["ground_truth"].get(f) is not None:
            print(f"\nExample mismatch in {r['id']} field={f}:")
            print(f"  pred: {r['extracted'].get(f)!r}")
            print(f"  true: {r['ground_truth'].get(f)!r}")
            break
    else:
        continue
    break


Field-level metrics over 20 invoices:

         field  support  precision  recall
invoice_number       20        1.0     1.0
  invoice_date       20        1.0     1.0
   vendor_name       20        1.0     1.0
 customer_name       20        1.0     1.0
      due_date       20        0.0     0.0
      subtotal       20        1.0     1.0
           tax       20        1.0     1.0
         total       20        1.0     1.0

Example mismatch in INV-2024000 field=due_date:
  pred: None
  true: '2026-05-23'


## Takeaways

Read the per-field precision/recall table above rather than any number I'd hand-wave here. A few things that stood out from this run:

- **The due_date column is recall=0.** Looking at the data, the truncation `text[:3000]` plus the way some invoices put the due date near the bottom probably drops that line before the model ever sees it. So this isn't really the LLM being wrong, it's the prompt cutting off the relevant text. Easy fix: either bump the truncation limit or only truncate the body, not the header/footer.
- The Pydantic + retry loop is doing real work. Even with `response_format=json_object` the model sometimes returns numbers as strings (`"$739.45"`) or skips a field. The `parse_currency` validator handles most of those without a second model call.
- Line items are the obvious next failure mode. Strict equality on a list of dicts is unforgiving. A real eval would do fuzzy match on `description` and an exact match on `total`, or compute set-IoU on the line-item totals.
- The Mistral-7B LoRA fine-tune I originally planned isn't here. On templated synthetic invoices the zero-shot baseline is already strong, and fine-tuning would mostly overfit to the template. The right time to fine-tune is when there's a real document distribution (scanned PDFs, OCR'd noise, multi-page contracts) actually hurting zero-shot accuracy.
- For any real deployment, latency and cost matter. Current setup is one LLM call per doc, sequential. Easy wins: batch in parallel via `asyncio` and cache results keyed on text hash.